# Museum of Spatial Transcriptomics metadata

Download the selected Museum of Spatial Transcriptomics Google Sheet tabs, normalize them into the shared metadata schema, and deduplicate rows that appear in multiple tabs. Tab provenance is retained in `museum_source_tabs` and `museum_duplicate_count`.


In [ ]:
from pathlib import Path
import importlib

import pandas as pd

import cellxgene_quantification
importlib.reload(cellxgene_quantification)
from cellxgene_quantification import (
    MUSEUM_ST_SHEET_TABS,
    deduplicate_museum_spatial_transcriptomics,
    fetch_museum_spatial_transcriptomics,
    normalize_museum_spatial_transcriptomics,
)

OUTPUT_PATH = Path("museum_of_st_meta.csv")


In [ ]:
museum_raw = fetch_museum_spatial_transcriptomics(sheet_names=MUSEUM_ST_SHEET_TABS)

print(f"Fetched raw rows: {len(museum_raw):,}")
print("Rows by tab:")
print(museum_raw["museum_sheet_tab"].value_counts(dropna=False).to_string())

fetch_errors = museum_raw.attrs.get("fetch_errors", {})
if fetch_errors:
    print("\nFetch errors:")
    for tab, error in fetch_errors.items():
        print(f"- {tab}: {error}")

museum_raw.head()


In [ ]:
museum_spatial_meta_raw = normalize_museum_spatial_transcriptomics(museum_raw)
museum_spatial_meta = deduplicate_museum_spatial_transcriptomics(museum_spatial_meta_raw)

print(f"Normalized rows before deduplication: {len(museum_spatial_meta_raw):,}")
print(f"Rows after deduplication: {len(museum_spatial_meta):,}")
print(f"Duplicate rows collapsed: {len(museum_spatial_meta_raw) - len(museum_spatial_meta):,}")

duplicate_summary = (
    museum_spatial_meta
    .query("museum_duplicate_count > 1")
    [["dataset_id", "museum_duplicate_count", "museum_source_tabs", "dataset_title", "dataset_type", "accession"]]
    .sort_values("museum_duplicate_count", ascending=False)
)

print(duplicate_summary.head(20).to_string(index=False))
museum_spatial_meta.head()


In [ ]:
museum_spatial_meta.to_csv(OUTPUT_PATH, index=False)
print(f"Wrote {OUTPUT_PATH.resolve()}")
print(museum_spatial_meta.shape)
